Model

In [1]:
import anndata
import numpy as np
import pandas as pd
from scipy.sparse import spmatrix
import scanpy as sc
from sklearn.preprocessing import StandardScaler

from celltypist import models

from models.classifier import Classifier

import logging
logger = logging.getLogger("Data-Preprocessing")

Data Preprocessing

In [2]:
data_path = "./data/global.h5ad"
raw = anndata.read_h5ad(data_path)
logger.info("🍳 Preparing data before training")

old_to_new_class_map = {}
with open("/home/lagandua/projects/sven/pineapple/cell_mapping3.txt", "r") as file:
    for line in file:
        line = line.strip()
        split_line = line.split(":")
        old_classes = split_line[1].split(",")
        new_class = split_line[0]
        for c in old_classes:
            old_to_new_class_map[c] = new_class

# Create new class labels
label = "sven_classes_new"
raw.obs[label] = [
    old_to_new_class_map[c] for c in raw.obs["Predicted_labels_CellTypist"]
]

🍳 Preparing data before training


In [3]:
# Value counts of new class labels
logger.info("🍳 Value counts of new class labels"
            f"\n{raw.obs[label].value_counts()}")

🍳 Value counts of new class labels
sven_classes_new
Helper T cells                79707
Cytotoxic T cells             56916
B Cells                       47148
ILC                           37624
Macrophages                   24921
Monocytes                     23910
Regulatory T cells            12059
Innate-like T cells           11896
Plasma cells                   8882
T cells                        6755
Mast Cells                     3562
HSC/MPP                        3000
DC                             2425
Double-negative thymocytes     1790
Cycling cells                  1649
Memory T cells                 1470
Erythroid                       836
pDC                             835
MNP                             627
Promyelocytes                   569
B Cell lineage                  430
Granulocytes                    357
ETP                             313
Early MK                        313
Megakaryocytes precursor        240
Double-positive thymocytes      237
Megakaryocyt

In [4]:

# Only keep the classes that have 2000 or more cells
min_cells_per_class = 2000
classes = raw.obs[label].value_counts()
classes = classes[classes >= min_cells_per_class].index
logger.info(f"🍳 Classes with {min_cells_per_class} or more cells"
            f"\n{classes}")

🍳 Classes with 2000 or more cells
Index(['Helper T cells', 'Cytotoxic T cells', 'B Cells', 'ILC', 'Macrophages',
       'Monocytes', 'Regulatory T cells', 'Innate-like T cells',
       'Plasma cells', 'T cells', 'Mast Cells', 'HSC/MPP', 'DC'],
      dtype='object', name='sven_classes_new')


In [5]:

# label = "Predicted_labels_CellTypist"
indata = raw.X[raw.obs[label].isin(classes)]
genes = raw.var_names
labels = raw.obs[raw.obs[label].isin(classes)][label]

if isinstance(indata, pd.DataFrame):
    indata = indata.values
elif isinstance(indata, spmatrix):
    indata = indata.toarray()

labels = np.array(labels)
genes = np.array(genes)

# Filter
models.models_path = "/home/lagandua/projects/sven/pineapple/celltypist/.cache"
ct_model = models.Model.load(model = 'Immune_All_Low.pkl')
chosen_genes = np.array(
        list(set(ct_model.classifier.features).intersection(set(raw.var_names)))
    )
keep_model = np.isin(raw.var_names, chosen_genes)

flag = indata.sum(axis = 0) == 0
if flag.sum() > 0:
    logger.info(f"✂️ {flag.sum()} non-expressed are filtered out")
    keep = (~flag) & keep_model
    genes = genes[keep]
logger.info(f"🔬 Input data has {indata.shape[0]} cells and {(keep).sum()} genes")


✂️ 6738 non-expressed are filtered out
🔬 Input data has 136623 cells and 6124 genes


In [10]:
# Scaler (this takes forever to run)
logger.info(f"⚖️ Scaling input data")
scaler = StandardScaler(with_mean = True)
indata = scaler.fit_transform(indata[:, keep] if keep.sum() > 0 else indata)
indata[indata > 10] = 10

⚖️ Scaling input data


In [12]:

# Save preprocessed data into train and test sets (this also takes forever)
logger.info(f"📝 Saving preprocessed data into train and validation sets")

val_size = 0.1 # 10%
true_count = int(indata.shape[0] * val_size)
false_count = indata.shape[0] - true_count

val_flag = np.concatenate((np.full(true_count, True), np.full(false_count, False)))
np.random.shuffle(val_flag)

np.savez(f'./data/{label}/train_high_90_min2K.npz', data=indata[~val_flag], labels=labels[~val_flag], genes=genes)
np.savez(f'./data/{label}/val_high_10_min2K.npz', data=indata[val_flag], labels=labels[val_flag], genes=genes)

📝 Saving preprocessed data into train and validation sets
